# 18 — Durable state versus disposable cache
PostgreSQL records runs and approvals. Redis caches only a pure ROI number;
losing the cache cannot change authorization or the result.



**Advanced follow-up to lesson 10 (resilience).**


## Problem

Files and process memory cannot safely coordinate an API's approvals.


## Concept

PostgreSQL keeps durable records; Redis holds only disposable cache values.

PostgreSQL transactions make approval replay-safe. Redis only caches a pure example calculation.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume   ◀ YOU ARE HERE
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
run → PostgreSQL → durable approval; pure ROI → Redis cache or the same Python calculation
```

run → PostgreSQL approval record; ROI → Redis cache/fallback. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Calculate an ROI value with disposable Redis caching

Calculate an ROI value with disposable Redis caching.


In [ ]:
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.cache import cached_annual_roi
from workshop.persistence import configured_store

print("ROI scenario:", cached_annual_roi(45_000, 30_000))
print("PostgreSQL configured:", configured_store() is not None)



## Check the database only in a configured advanced environment
This does not create a run or send an email.



## Step 2 — Check configured PostgreSQL storage and its schema

Check configured PostgreSQL storage and its schema.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    store = configured_store()
    if store is None:
        raise RuntimeError("Set WORKSHOP_DATABASE_URL before running the live state lab")
    store.initialize()
    print("PostgreSQL ready:", store.ping())
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true to check PostgreSQL connectivity.")


## Failure mode to know about

Redis can disappear. Approval and action IDs belong in PostgreSQL, not only in a cache.


## Production implication

Use migrations and backups for PostgreSQL. Treat Redis outages as performance events.


## Your turn

Stop Redis and repeat the ROI calculation.

**Checkpoint:** Cache loss does not alter the result or approval state.
